In [1]:
import os

import numpy as np
import pandas as pd

from utils import PROCESSED_DIR

# Directory containing MPsizeBase
DATA_DIR = "data/original"

# Overwrite outputs
OVERWRITE = True


def load_mpsizebase(path: str) -> pd.DataFrame:
    """Load MPsizeBase"""

    # Load
    mpsb = pd.read_excel(path, sheet_name="Literature PSD data", header=2)
    mpsb.columns = mpsb.columns.str.lower()
    mpsb = mpsb.loc[mpsb["link"].notna()]  # skip any blank rows

    # Parse DOI from "link" column
    mpsb["doi"] = (
        mpsb["link"].str.strip().str.replace(r"^https?://doi.org/", "", regex=True)
    )

    # Merge "study" and "year" columns
    mpsb["study"] = mpsb["study"].apply(lambda x: x.split(" ")[0])
    mpsb["study"] = mpsb["study"] + mpsb["year"].astype(str)

    # Subset columns
    mpsb = mpsb[
        [
            "study",
            "doi",
            "lat",
            "lon",
            "alt",
            "sample_type",
            "sub_type1",
            "sub_type2",
            "psd_id",
            "mp_shape",
            "mp_density",
            "low_size",
            "high_size",
            "cum_sum",
            "cum_sum_units",
            "filtered_cum_sum",
            "cumsum_slope",
            "cumsum_log_intercept",
            "cumsum_powerlaw_alpha",
            "cumsum_powerlaw_scale",
            "shape_factor",
            "comment",
        ]
    ]

    # Rename columns
    mpsb = mpsb.rename(
        columns=lambda x: (
            x.removeprefix("mp_").removeprefix("cumsum_").removeprefix("powerlaw_")
        )
    )

    return mpsb


def get_atmospheric_studies(mpsb: pd.DataFrame) -> pd.DataFrame:
    """Return studies of atmospheric microplastics"""

    mpsb = (
        mpsb.loc[
            mpsb["sample_type"].eq("Atmosphere")
            # Exclude peat archive samples (unclear timeframe)
            & mpsb["sub_type1"].isin(
                ["Aerosol", "Total deposition", "Dry deposition", "Wet deposition"]
            )
        ]
        .reset_index(drop=True)
    )

    # Harmonize sub_type2
    mpsb["sub_type2"] = (
        mpsb["sub_type2"].replace("Rural", "Outdoor").replace("Urban", "Outdoor")
    )

    return mpsb


def merge_wet_dry_deposition(data: pd.DataFrame) -> pd.DataFrame:
    """Merge colocated wet and dry deposition giving total deposition"""

    def merge_psd_ids(x: pd.Series) -> str:
        """Merge two PSD IDs e.g. Bra20_A_1 + Bra20_B_1 -> Bra20_AB_1"""

        letters = "".join(sorted(set(x.str[6])))
        psd_id = x.replace(f"_[{letters}]_", f"_{letters}_", regex=True).unique()[0]
        return psd_id

    def fit_powerlaw(x: pd.DataFrame) -> pd.Series:
        """Fit a powerlaw to log-transformed low size and cumulative counts

        Derivation [NOTE: MPsizeBase records alpha < 0]:
                  n(x) = C * x^alpha
            N(X>=xmin) = Integral[C * x^alpha] dx
                       = -C / (alpha + 1) * xmin^(alpha + 1)
                log(N) = log[-C / (alpha + 1)] + (alpha + 1) * log(xmin)
                       = intercept + slope * log(xmin)
                 alpha = slope - 1
                     C = -exp(intercept) * (alpha + 1)
        """

        # Fit slope and regression to log-transformed data
        x_log = np.log10(x["low_size"])
        y_log = np.log10(x["cum_sum"])
        slope, log_intercept = np.polyfit(x_log, y_log, deg=1)

        # Compute powerlaw parameters 'alpha' and 'scale'
        alpha = slope - 1
        scale = -(10**log_intercept) * (alpha + 1)

        return pd.Series(
            {
                "slope": slope,
                "log_intercept": log_intercept,
                "alpha": alpha,
                "scale": scale,
            }
        )

    # Sum co-located wet + dry deposition in matching size bins -> total deposition
    data = (
        data.loc[data["sub_type1"].isin(["Dry deposition", "Wet deposition"])]
        .groupby(
            ["study", "doi", "lat", "lon", "sub_type2", "shape", "low_size", "high_size"],
            as_index=False,
            sort=False,
        )
        .agg(
            psd_id=("psd_id", merge_psd_ids),
            density=("density", "first"),
            shape_factor=("shape_factor", "first"),
            cum_sum=("cum_sum", "sum"),
            n_counts=("cum_sum", "count"),
            n_unfiltered=("filtered_cum_sum", "count"),
        )
        .assign(sub_type1="Total deposition")
    )

    # Sanity check: ensure all size bins had counts for both wet and dry deposition
    if data["n_counts"].ne(2).any():
        raise ValueError("All size bins must have counts for both wet and dry deposition")

    # Fit powerlaw to each cumulative particle size distribution, using only size bins
    # that MPsizeBase used to fit powerlaw for both wet and dry deposition.
    params = (
        data.loc[data["n_unfiltered"].eq(2)]
        .groupby(["study", "doi", "psd_id"], sort=False)
        .apply(fit_powerlaw, include_groups=False)
    )

    # Collapse data to one row per observation and add columns with powerlaw parameters
    data = summarize_mpsb(data).join(params, on=params.index.names)

    return data


def summarize_mpsb(data: pd.DataFrame) -> pd.DataFrame:
    """Return study, location, size range, and powerlaw parameters for each PSD"""

    # fmt: off
    group_cols = [
        "study", "doi", "lat", "lon", "sub_type1", "sub_type2", "psd_id", "shape"
    ]
    # fmt: on
    aggs = {
        "low_size": "min",
        "high_size": "max",
        "density": "first",
        "shape_factor": "first",
        "slope": "first",
        "log_intercept": "first",
        "alpha": "first",
        "scale": "first",
    }
    aggs = {k: v for k, v in aggs.items() if k in data}

    return data.groupby(group_cols, as_index=False, sort=False).agg(aggs)


Introduction
------------

This notebook processes data from MPsizeBase and computes a typical size distribution for atmospheric microplastics.

[MPsizeBase](https://doi.org/10.5281/zenodo.18201301) [@Sonke2026; @Segur2026] is a large database of size-resolved environmental microplastic observations collected from the literature. It reports the parameters of a power law that best fit each observation's particle size distribution. We compute the median value of the parameter $\alpha$, which describes the relative frequency of small vs. large particles.

Preprocess MPsizeBase
---------------------

Most studies measured aerosols or total deposition. However, one study (@Brahney2020) simultaneously measured wet and dry deposition and reported them separately. To make the data comparable with other studies, we combine the wet and dry deposition counts to get total deposition and re-fit the parameters of the power law size distribution.

In [2]:
def process_mpsizebase(
    mpsb_name: str,
    mpsb_dir: str,
    out_name: str,
    out_dir: str = PROCESSED_DIR,
    overwrite: bool = False,
) -> pd.DataFrame:
    """Preproces MPsizeBase data"""

    out_path = os.path.join(out_dir, out_name)
    if os.path.exists(out_path) and not overwrite:
        print(f"File exists: {out_path}")
        return pd.read_csv(out_path)

    # Load data
    mpsb_path = os.path.join(mpsb_dir, mpsb_name)
    mpsb = load_mpsizebase(mpsb_path)

    # Select only studies of atmospheric microplastic concentration or deposition
    mpsb = get_atmospheric_studies(mpsb)

    # Compute total deposition from Brahney2020's colocated measures of wet + dry
    # deposition and refit powerlaw
    brahney = mpsb.loc[mpsb["doi"].eq("10.1126/science.aaz5819")]
    brahney = merge_wet_dry_deposition(brahney)

    # Summarize size range and powerlaw parameters for each observation
    mpsb = summarize_mpsb(
        mpsb.loc[mpsb["sub_type1"].isin(["Aerosol", "Total deposition"])]
    )

    # Combine with total deposition of Brahney2020
    mpsb = (
        pd.concat([mpsb, brahney])
        .sort_values(["study", "doi", "psd_id"])
        .reset_index(drop=True)
    )

    # Identify unique particle size distributions
    mpsb.insert(
        loc=mpsb.columns.get_loc("psd_id"),
        column="psd",
        value=mpsb["psd_id"].replace(r"_\d+$", "", regex=True),
    )

    # Save as CSV
    mpsb.to_csv(out_path, index=False)
    print(f"-> {out_path}")

    return mpsb


mpsb = process_mpsizebase(
    mpsb_name="MPsizeBase v9-1-2026.xlsx",
    mpsb_dir=DATA_DIR,
    out_name="mpsizebase_atmo.csv",
    overwrite=OVERWRITE,
)

-> data/processed/mpsizebase_atmo.csv


Summary
-------

We compute summary statistics (median, min, max, mean, standard deviation) for the power law $\alpha$ parameter (exponent) across all observations in MPsizeBase, grouping observations by sample type (aerosol vs. total deposition), indoor/outdoor, and microplastic shape (fibers vs. fragments).

In [3]:
# Summarize for reference
# Aggregate over observations by sample type, shape, and indoor/outdoor
(
    mpsb.loc[mpsb["shape"].isin(["Fibers", "Fragments"])]
    .groupby(["sub_type1", "shape", "sub_type2"])
    .agg(
        {
            "doi": "nunique",
            "psd": "nunique",
            "psd_id": "nunique",
            "alpha": ["median", "min", "max", "mean", "std"],
        }
    )
    .rename(columns={"psd_id": "obs"})
    .round(2)
)

doi     psd     obs  alpha        \
                                     nunique nunique nunique median   min   
sub_type1        shape     sub_type2                                        
Aerosol          Fibers    Indoor          1       1       1  -2.44 -2.44   
                           Outdoor         8       8      10  -1.67 -2.04   
                 Fragments Indoor          6       6       6  -2.83 -3.24   
                           Outdoor         8       8      16  -2.83 -4.31   
Total deposition Fibers    Outdoor         9       9      19  -1.76 -2.28   
                 Fragments Outdoor         7      12      22  -2.39 -3.14   

                                                        
                                       max  mean   std  
sub_type1        shape     sub_type2                    
Aerosol          Fibers    Indoor    -2.44 -2.44   NaN  
                           Outdoor   -1.40 -1.72  0.19  
                 Fragments Indoor    -2.13 -2.72  0.45  
                           Outdoor   -1.94 -2.62  0.62  
Total deposition Fibers    Outdoor   -1.59 -1.83  0.21  
                 Fragments Outdoor   -1.89 -2.40  0.30

Because some studies sampled at multiple locations but reported a single aggregate particle size distribution, it may make sense compute the summary statistics with an equal weight for each unique size distribution (rather than weighting each observation equally):

In [4]:
# Summarize for reference
# Aggregate over PSDs by sample type, shape, and indoor/outdoor
(
    mpsb.loc[mpsb["shape"].isin(["Fibers", "Fragments"])]
    .groupby(["doi", "sub_type1", "shape", "sub_type2", "psd"], as_index=False)[["alpha"]]
    .mean()
    .groupby(["sub_type1", "shape", "sub_type2"])
    .agg(
        {
            "doi": "nunique",
            "psd": "nunique",
            "alpha": ["median", "min", "max", "mean", "std"],
        }
    )
    .round(2)
)

doi     psd  alpha                    \
                                     nunique nunique median   min   max  mean   
sub_type1        shape     sub_type2                                            
Aerosol          Fibers    Indoor          1       1  -2.44 -2.44 -2.44 -2.44   
                           Outdoor         8       8  -1.72 -2.04 -1.40 -1.73   
                 Fragments Indoor          6       6  -2.83 -3.24 -2.13 -2.72   
                           Outdoor         8       8  -2.83 -4.31 -1.94 -2.80   
Total deposition Fibers    Outdoor         9       9  -2.03 -2.28 -1.59 -1.96   
                 Fragments Outdoor         7      12  -2.33 -3.14 -1.89 -2.42   

                                            
                                       std  
sub_type1        shape     sub_type2        
Aerosol          Fibers    Indoor      NaN  
                           Outdoor    0.21  
                 Fragments Indoor     0.45  
                           Outdoor    0.74  
Total deposition Fibers    Outdoor    0.22  
                 Fragments Outdoor    0.41